---
# `Movie Recommendation System`
---

In [ ]:
import pandas as pd
import numpy as np

### Data Gathering

In [ ]:
movies = pd.read_csv("../Datasets/tmdb_5000_movies.csv")

In [ ]:
credits = pd.read_csv('../Datasets/tmdb_5000_credits.csv')

### Data Exploration

In [ ]:
movies.head(3)

Important points
1. Budge: give infor about the budget that it took in creating this movie
2. Genres: what type of movie
3. Id: id of this movie in the tmdb website
4. Keywords: what type of movie it is
5. Original title: title in regional language 
6. overview: description of the movie

In [ ]:
movies.describe()

In [ ]:
movies[movies['popularity'] == 875.581305]

In [ ]:
credits.head(3)

In [ ]:
data = movies.merge(credits, on='title')

In [ ]:
data.head(3)

In [ ]:
data['genres']

In [ ]:
data['genres'][0]

In [ ]:
data.columns

In [ ]:
data = data[['genres', 'id', 'keywords', 'overview', 'release_date',  'title', 'cast', 'crew']]

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.shape

In [ ]:
data.duplicated().sum()

### `Data Preprocessing`

In [ ]:
data['genres']

In [ ]:
data.genres[0]

In [ ]:
# Retrieve the value in such a way such that for every row of the column

def genre_extract(col):
    L = []

    for i in col:
        L.append(i['name'])
    
    return L

In [ ]:
genre_extract([{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}])

In [ ]:
# data['genres'].apply(genre_extract) # Through error

In [ ]:
import ast

In [ ]:
# Retrieve the value in such a way such that for every row of the column

def genre_extract(col):
    L = []

    for i in ast.literal_eval(col):
        L.append(i['name'])
    
    return L

In [ ]:
data['genres'] = data['genres'].apply(genre_extract)

In [ ]:
data.head()

In [ ]:
def extract_keyword(col):
    L = []

    for i in ast.literal_eval(col):
        L.append(i['name'])
    
    return L

In [ ]:
data['keywords'] = data['keywords'].apply(extract_keyword)

In [ ]:
data['release_date'] = pd.to_datetime(data['release_date'])

In [ ]:
data['release_date'] = data['release_date'].dt.year

In [ ]:
data.head()

In [ ]:
def cast_extract(col):
    L = []
    j = 1
    for i in ast.literal_eval(col):
        if j <=3:
            L.append(i['name'])
            j+=1
    
    return L

In [ ]:
data['cast'] = data['cast'].apply(cast_extract)

In [ ]:
data.head(3)

In [ ]:
def crew_extract(col):
    L = []
    for i in ast.literal_eval(col):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    
    return L

In [ ]:
data['crew'] = data['crew'].apply(crew_extract)

In [ ]:
data.head()

In [ ]:
data['overview'] = data['overview'].apply(lambda x: x.split())

In [ ]:
data.head()

In [ ]:
data['cast'] = data['cast'].apply(lambda x: [i.replace(" " , "") for i in x])
data['crew'] = data['crew'].apply(lambda x: [i.replace(" " , "") for i in x])
data['keywords'] = data['keywords'].apply(lambda x: [i.replace(" " , "") for i in x])
data['genres'] = data['genres'].apply(lambda x: [i.replace(" " , "") for i in x])

In [ ]:
data.head()

In [ ]:
data['tags'] = data['genres'] + data['keywords'] + data['cast'] + data['crew'] + data['overview']

In [ ]:
data.head()

In [ ]:
data = data.drop(columns=['genres', 'keywords', 'cast', 'release_date', 'crew', 'overview'])

In [ ]:
data.head()

In [ ]:
data['tags'] = data['tags'].apply(lambda x: ' '.join(x))

In [ ]:
data.head()

### `lowecasing`

In [ ]:
data['tags'] = data['tags'].apply(lambda x: x.lower())
data['title'] = data['title'].apply(lambda x: x.lower())

In [ ]:
data.head()

In [ ]:
data['tags'][0]

### `NLP Preprocessing and Feature Engineering`

Text vectorization: text to numerical format to compare other numerical movies

Move will be the distance between the points then less similar the movie

Bag Of Words: concat all tags of movies : most common or repeated words in the collection of these 4805 moives tags

Before Bag of words
1. Stop word Removal - of, they, is , an ... etc
   1. Doesn't contribute in the meaning or content of the sentence but only contribute in the sentence formation
   2. These vector representate consume memory
2. Stemming : similar type of word doesn't get repeated
   1. required when represent the root words 
3. Then apply, Bag of Words


Stop word removal --> stemming --> bag of word or word2vec --> cosine similarity of with rest movie --> function for movie recommendation

are required

In [ ]:
data.shape

### `Stemming`
- library used for nlp: nltk() --> natural language tool kit --> almost all function

In [ ]:
# !pip install nltk

In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
ps = PorterStemmer()

In [ ]:
data['tags'][0]

In [ ]:
def stemming(text):
    L= []

    for i in text.split():
        L.append(ps.stem(i))
    
    return " ".join(L)

In [ ]:
data['tags'] = data['tags'].apply(stemming)

In [ ]:
data['tags'][0]

Vectorizatoin: countvectorizer() --> parameter -> stop-word='english', max_features = 5000

from sklearn.feature_extraction.text import CountVectorizer 

cv.get_feature_names_out() --> what common features are there

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
vectors = cv.fit_transform(data['tags'])

In [ ]:
vectors = vectors.toarray()

In [ ]:
vectors

In [ ]:
cv.get_feature_names_out()[:200]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarities = cosine_similarity(vectors)

In [ ]:
similarities

In [ ]:
similarities[0]

In [ ]:
data[data['title'] == 'avatar'].index[0]

In [ ]:
def recommend(movie):
    movie_index = data[data['title'] == movie].index[0]
    recommendations = similarities[movie_index]
    movie_list = sorted(enumerate(recommendations), reverse=True, key=lambda x: x[1])[1:6]

    for i in movie_list:
        print(data.iloc[i[0]].title)


In [ ]:
recommend('avatar')